# E-commerce Return Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether an e-commerce order will be **returned** based on order details: product category, price, discount, delivery time, customer history, and payment method. Synthetic dataset with ~8,000 rows and ~25% return rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given order characteristics (product category, price, discount, delivery days, customer history, payment method), predict whether the order will be returned (returned = 1) or kept (returned = 0).

## 5 · Why This Project Matters

- E-commerce returns cost retailers **$800+ billion annually** worldwide.
- Predicting returns enables proactive quality checks and logistics optimization.
- This teaches working with **mixed feature types** and moderate class imbalance.
- Return rate optimization directly impacts profitability.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~8,000 |
| Features | 10 (product_category, price, discount_pct, delivery_days, customer_orders, customer_returns, payment_method, product_rating, size_mismatch_flag, gift_flag) |
| Target | `returned` (binary: 0=kept, 1=returned) |
| Class balance | ~75% kept, ~25% returned |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic e-commerce return dataset.
**License**: Educational / synthetic.
**Note**: Simulates online retail return patterns.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "returned"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\E-commerce Return Prediction


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 8000

product_category = np.random.choice(['clothing', 'electronics', 'home', 'beauty', 'sports', 'books'], n,
                                      p=[0.3, 0.2, 0.15, 0.15, 0.1, 0.1])
price = np.random.lognormal(3.5, 0.8, n).clip(5, 500).round(2)
discount_pct = np.random.choice([0, 5, 10, 15, 20, 25, 30, 40, 50], n,
                                 p=[0.3, 0.1, 0.15, 0.1, 0.1, 0.08, 0.07, 0.05, 0.05])
delivery_days = np.random.poisson(5, n).clip(1, 21)
customer_orders = np.random.poisson(8, n).clip(1, 50)
customer_returns = np.random.binomial(customer_orders, 0.15)
payment_method = np.random.choice(['credit_card', 'debit_card', 'paypal', 'cod'], n,
                                    p=[0.4, 0.25, 0.2, 0.15])
product_rating = np.random.uniform(1, 5, n).round(1)
size_mismatch = np.random.choice([0, 1], n, p=[0.85, 0.15])
gift_flag = np.random.choice([0, 1], n, p=[0.9, 0.1])

# Return probability based on features
return_prob = (
    0.15 * (product_category == 'clothing').astype(float)
    + 0.05 * discount_pct / 50
    + 0.02 * delivery_days / 7
    + 0.1 * customer_returns / (customer_orders + 1)
    - 0.05 * (product_rating - 3) / 2
    + 0.25 * size_mismatch
    + 0.05 * (payment_method == 'cod').astype(float)
    - 0.03 * gift_flag
    + np.random.normal(0, 0.15, n)
).clip(0, 1)
returned = (return_prob > 0.25).astype(int)

df = pd.DataFrame({
    'product_category': product_category, 'price': price,
    'discount_pct': discount_pct, 'delivery_days': delivery_days,
    'customer_orders': customer_orders, 'customer_returns': customer_returns,
    'payment_method': payment_method, 'product_rating': product_rating,
    'size_mismatch_flag': size_mismatch, 'gift_flag': gift_flag,
    'returned': returned,
})
print(f"Dataset shape: {df.shape}")
print(f"Return rate: {df['returned'].mean():.2%}")
df.head()

Dataset shape: (8000, 11)
Return rate: 24.60%


,product_category,price,discount_pct,delivery_days,customer_orders,customer_returns,payment_method,product_rating,size_mismatch_flag,gift_flag,returned
0,electronics,71.31,0,4,3,0,credit_card,1.4,0,0,0
1,books,81.63,0,7,10,0,credit_card,1.5,1,0,1
2,beauty,6.97,5,2,8,1,debit_card,2.8,0,0,0
3,home,72.44,15,5,6,1,debit_card,2.0,0,0,0
4,clothing,17.32,0,2,7,1,cod,2.1,0,0,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (8000, 11)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
returned
0    6032
1    1968
Name: count, dtype: int64

Dtypes:
product_category       object
price                 float64
discount_pct            int64
delivery_days           int32
customer_orders         int32
customer_returns        int32
payment_method         object
product_rating        float64
size_mismatch_flag      int64
gift_flag               int64
returned                int64
dtype: object

Target 'returned' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(f"{col} by Return")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['product_category', 'payment_method']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Return Rate by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 8, Categorical: 2


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Return Distribution")
axes[0].set_xticklabels(['Kept (0)', 'Returned (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

Class distribution:
returned
0    6032
1    1968
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (6400, 10), Test: (1600, 10)
Train target dist: {0: 4826, 1: 1574}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
X_train = X_train.copy(); X_test = X_test.copy()
X_train['return_rate'] = X_train['customer_returns'] / (X_train['customer_orders'] + 1)
X_test['return_rate'] = X_test['customer_returns'] / (X_test['customer_orders'] + 1)
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (11): ['product_category', 'price', 'discount_pct', 'delivery_days', 'customer_orders', 'customer_returns', 'payment_method', 'product_rating', 'size_mismatch_flag', 'gift_flag', 'return_rate']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")


BASELINE: Logistic Regression
Accuracy : 0.8087
Precision: 0.7946
Recall   : 0.8087
F1       : 0.7930
ROC-AUC  : 0.7359


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
GaussianNB                     0.808750           0.683448  0.747746  0.793623   0.794696  0.808750    0.018205
NearestCentroid                0.790000           0.682973  0.731484  0.781506   0.777705  0.790000    0.020431
QuadraticDiscriminantAnalysis  0.804375           0.682255  0.744088  0.790475   0.789828  0.804375    0.021030
LinearSVC                      0.808125           0.682179  0.735390  0.792805   0.793911  0.808125    0.023623
RidgeClassifier                0.808125           0.682179  0.735407  0.792805   0.793911  0.808125    0.019435
SGDClassifier                  0.808125           0.682179  0.706895  0.792805   0.793911  0.808125    0.053277
RidgeClassifierCV              0.808125           0.682179  0.735367  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.7994
F1       : 0.7909


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test).ravel()
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8019  F1: 0.7855


LightGBM  Acc: 0.7881  F1: 0.7754


XGBoost   Acc: 0.7950  F1: 0.7828


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.8087  0.7930     0.7946  0.8087
FLAML                  0.7994  0.7909     0.7877  0.7994
CatBoost               0.8019  0.7855     0.7861  0.8019
XGBoost                0.7950  0.7828     0.7801  0.7950
LightGBM               0.7881  0.7754     0.7720  0.7881

Top 3: ['Logistic Regression', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  Logistic Regression
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      1206
           1       0.67      0.43      0.53       394

    accuracy                           0.81      1600
   macro avg       0.75      0.68      0.70      1600
weighted avg       0.79      0.81      0.79      1600


  FLAML
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1206
           1       0.62      0.49      0.54       394

    accuracy                           0.80      1600
   macro avg       0.73      0.69      0.71      1600
weighted avg       0.79      0.80      0.79      1600


  CatBoost
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      1206
           1       0.65      0.42      0.51       394

    accuracy                           0.80      1600
   macro avg       0.74      0.67      0.6

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: Logistic Regression
Error rate: 0.1913 (306 / 1600)

Errors by true class:
      errors  total  error_rate
true                           
0         82   1206    0.067993
1        224    394    0.568528


## 25 · Interpretation and Business Insight

- **Size mismatch flag** is the strongest return predictor — sizing issues drive clothing returns.
- **Customer return history** (return rate) predicts habitual returners.
- **Product category** matters — clothing has the highest return rate.
- **Product rating** inversely correlates — better-rated products are returned less.
- **Delivery time** has a small positive effect on returns.

## 26 · Limitations

1. Synthetic data — real returns depend on product photos, descriptions, sizing.
2. No text features (return reason, product reviews).
3. 'Size mismatch' is a post-purchase signal, not available pre-order.
4. No product image quality or description accuracy features.
5. Missing seasonality (holiday returns spike).

## 27 · How to Improve This Project

1. Add return reason text for NLP analysis.
2. Include product review sentiment and photo quality.
3. Engineer customer lifetime value and return cost.
4. Build category-specific models (clothing vs electronics).
5. Add competitor pricing and seasonal features.

## 28 · Production Considerations

- Pre-shipment return risk scoring for quality checks.
- Dynamic return policies based on predicted risk.
- Customer-specific recommendations to reduce returns.
- Integration with warehouse and logistics systems.

## 29 · Common Mistakes

1. Including post-purchase features (size_mismatch) in pre-order prediction.
2. Not computing cost-of-return for business optimization.
3. Treating all returns equally (some are exchanges, not refunds).
4. Not segmenting by product category.
5. Using accuracy when return rate is ~25% (class imbalance).

## 30 · Mini Challenge / Exercises

1. Remove size_mismatch_flag (post-purchase) and retrain — how much accuracy drops?
2. Calculate the cost savings of catching 50% of returns early.
3. Build a clothing-only model — does it perform better?
4. Find the optimal threshold for balancing precision and recall.

## 31 · Final Summary / Key Takeaways

1. E-commerce return prediction is a high-value retail ML problem.
2. Size mismatch and customer return history are the strongest signals.
3. Clothing has the highest return rate — category-specific models help.
4. Pre-order vs post-order feature availability matters for deployment.
5. Cost-sensitive evaluation is more valuable than accuracy.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8019,
    "f1": 0.7855,
    "precision": 0.7861,
    "recall": 0.8019
  },
  "LightGBM": {
    "accuracy": 0.7881,
    "f1": 0.7754,
    "precision": 0.772,
    "recall": 0.7881
  },
  "XGBoost": {
    "accuracy": 0.795,
    "f1": 0.7828,
    "precision": 0.7801,
    "recall": 0.795
  },
  "Logistic Regression": {
    "accuracy": 0.8087,
    "f1": 0.793,
    "precision": 0.7946,
    "recall": 0.8087
  },
  "FLAML": {
    "accuracy": 0.7994,
    "f1": 0.7909,
    "precision": 0.7877,
    "recall": 0.7994
  }
}
